# FOLIO test and FOLIO control: analysis

This notebook reproduces the statistics reported for the FOLIO test and the FOLIO control (Results and Methods).
It reads the released response exports in `data/folio/responses/`, the question files in `data/folio/*_test/`,
and the anonymous inclusion flags in `data/participant_inclusion.csv`.

All tests of individual accuracy take participants as the unit of analysis.

In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

LETTERS = {"A": 0, "B": 1, "C": 2, "D": 3}
TESTS = {
    "FOLIO test":    ("data/folio/responses/folio_actual_test_responses.csv",  "data/folio/actual_test/folio_actual_test.json"),
    "FOLIO control": ("data/folio/responses/folio_control_test_responses.csv", "data/folio/control_test/folio_control_test.json"),
}
inclusion = pd.read_csv("data/participant_inclusion.csv", keep_default_na=False)

def first(v):
    return v[0] if isinstance(v, list) and v else v

def load_long(name):
    """One row per included participant x main (non-attention-check) question."""
    responses_path, test_path = TESTS[name]
    test = json.load(open(test_path))
    attention = {i + 1 for i in test["attention_check_indices"]}           # question numbers are 1-based
    questions = {i + 1: q for i, q in enumerate(test["questions"])}
    responses = pd.read_csv(responses_path)
    completed = responses[responses["is_complete"].str.lower() == "yes"]
    flags = inclusion[inclusion["study"] == name].set_index("record")["included"]
    rows = []
    for _, r in completed.iterrows():
        if not flags["response_row_id %d" % r["response_row_id"]]:
            continue
        answers, times = json.loads(r["answers_json"]), json.loads(r["time_per_question_s"])
        for qnum, q in questions.items():
            if qnum in attention:
                continue
            answer = first(answers.get(str(qnum)))
            rows.append(dict(
                participant=int(r["response_row_id"]), question=qnum, model=q["model_name"],
                problem=str(q.get("story_key", q.get("story_id"))),
                correct=int(answer is not None and LETTERS.get(str(answer).upper()) == LETTERS[first(q["correct_answers"])]),
                rt=times.get(str(qnum))))
    return pd.DataFrame(rows), len(completed)

data, n_completed = {}, {}
for name in TESTS:
    data[name], n_completed[name] = load_long(name)
    d = data[name]
    print(f"{name}: {n_completed[name]} completed questionnaires, {d.participant.nunique()} analysed participants, "
          f"{d.question.nunique()} main questions from {d.problem.nunique()} problems, {len(d)} responses")

FOLIO test: 84 completed questionnaires, 80 analysed participants, 28 main questions from 19 problems, 2240 responses
FOLIO control: 82 completed questionnaires, 80 analysed participants, 28 main questions from 11 problems, 2240 responses


## 1. Individual accuracy (Results)

In [2]:
def mean_ci(x):
    x = np.asarray(x, float)
    half = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return x.mean(), x.mean() - half, x.mean() + half

accuracy = {}
for name, d in data.items():
    acc = d.groupby("participant").correct.mean() * 100        # percent correct per participant
    accuracy[name] = acc
    m, lo, hi = mean_ci(acc)
    q1, q3 = np.percentile(acc, [25, 75])
    w = stats.wilcoxon(acc - 25)                                 # vs 25% chance
    print(f"{name}: mean {m:.1f}% (SD {acc.std():.1f}%, 95% CI [{lo:.1f}%, {hi:.1f}%]), median {acc.median():.1f}%, IQR {q3 - q1:.1f} pp; "
          f"Wilcoxon W = {w.statistic:.1f}, p = {w.pvalue:.2g}")
    print(f"   participants below 50%: {(acc < 50).sum()}, at 50%: {np.isclose(acc, 50).sum()}, above 50%: {(acc > 50 + 1e-9).sum()}")

f, c = accuracy["FOLIO test"], accuracy["FOLIO control"]
u = stats.mannwhitneyu(c, f)
d_cohen = (c.mean() - f.mean()) / np.sqrt((c.var() + f.var()) / 2)
print(f"\nFOLIO test vs FOLIO control: Mann-Whitney U = {u.statistic:.1f}, p = {u.pvalue:.2g}; Cohen's d = {d_cohen:.2f}")

FOLIO test: mean 35.5% (SD 7.6%, 95% CI [33.8%, 37.2%]), median 35.7%, IQR 10.7 pp; Wilcoxon W = 27.0, p = 4.2e-13
   participants below 50%: 75, at 50%: 3, above 50%: 2
FOLIO control: mean 70.8% (SD 17.1%, 95% CI [67.0%, 74.6%]), median 75.0%, IQR 17.9 pp; Wilcoxon W = 9.0, p = 1.5e-14
   participants below 50%: 8, at 50%: 2, above 50%: 70

FOLIO test vs FOLIO control: Mann-Whitney U = 5961.5, p = 3.6e-21; Cohen's d = 2.66


## 2. Model comparison in the FOLIO test (Results; Methods: Wilcoxon signed-rank test, paired 95% CI, participant-clustered GEE)

In [3]:
d = data["FOLIO test"].copy()
d["r1"] = d.model.str.contains("DeepSeek", case=False).astype(int)
per_model = d.groupby(["participant", "r1"]).correct.mean().unstack() * 100
diff = per_model[1] - per_model[0]
m, lo, hi = mean_ci(diff)
w = stats.wilcoxon(per_model[1], per_model[0])
print(f"Qwen-3 items: {per_model[0].mean():.1f}% (SD {per_model[0].std():.1f}%); DeepSeek-R1 items: {per_model[1].mean():.1f}% (SD {per_model[1].std():.1f}%)")
print(f"Difference {m:.1f} pp, paired 95% CI [{lo:.1f}, {hi:.1f}]; Wilcoxon W = {w.statistic:.1f}, p = {w.pvalue:.2g}")

gee = smf.gee("correct ~ r1", groups="participant", data=d, family=sm.families.Binomial()).fit()
ci = np.exp(gee.conf_int().loc["r1"])
print(f"Participant-clustered GEE logistic regression: odds ratio (DeepSeek-R1 vs Qwen-3) = {np.exp(gee.params['r1']):.2f}, "
      f"95% CI [{ci[0]:.2f}, {ci[1]:.2f}], p = {gee.pvalues['r1']:.2g}")

Qwen-3 items: 31.6% (SD 11.8%); DeepSeek-R1 items: 39.5% (SD 9.4%)
Difference 7.9 pp, paired 95% CI [4.6, 11.2]; Wilcoxon W = 397.5, p = 4.7e-05
Participant-clustered GEE logistic regression: odds ratio (DeepSeek-R1 vs Qwen-3) = 1.41, 95% CI [1.22, 1.63], p = 2.9e-06


## 3. Response time (Results; Methods: responses under 5 s or above Q3 + 3 x IQR are set aside for this analysis only)

A few answers were saved without a separate response time; they count for accuracy and are absent only here.

In [4]:
for name, d in data.items():
    timed = d[d.rt.notna()]
    q1, q3 = timed.rt.quantile([0.25, 0.75])
    upper = q3 + 3 * (q3 - q1)
    kept = timed[(timed.rt >= 5) & (timed.rt <= upper)]
    removed = len(timed) - len(kept)
    r, p = stats.pearsonr(kept.rt, kept.correct)
    print(f"{name}: {len(d)} responses, {len(timed)} with a recorded time; set aside {removed} ({100 * removed / len(timed):.1f}%: "
          f"{(timed.rt < 5).sum()} under 5 s, {(timed.rt > upper).sum()} over {upper:.1f} s); analysed {len(kept)}")
    print(f"   mean response time {kept.rt.mean():.1f} s (SD {kept.rt.std():.1f}); Pearson r with correctness = {r:.3f}, p = {p:.2g}")

FOLIO test: 2240 responses, 2236 with a recorded time; set aside 36 (1.6%: 1 under 5 s, 35 over 320.9 s); analysed 2200
   mean response time 84.1 s (SD 56.3); Pearson r with correctness = -0.035, p = 0.1
FOLIO control: 2240 responses, 2235 with a recorded time; set aside 81 (3.6%: 1 under 5 s, 80 over 212.2 s); analysed 2154
   mean response time 52.2 s (SD 38.7); Pearson r with correctness = -0.135, p = 3.2e-10


## 4. Internal consistency (Results)

In [5]:
for name, d in data.items():
    M = d.pivot(index="participant", columns="question", values="correct").values
    k = M.shape[1]
    alpha = k / (k - 1) * (1 - M.var(axis=0, ddof=1).sum() / M.sum(axis=1).var(ddof=1))
    print(f"{name}: Cronbach's alpha = {alpha:.2f}")

FOLIO test: Cronbach's alpha = -0.06
FOLIO control: Cronbach's alpha = 0.84
